# CNN-EXP-017 — BPE-200 Global coarse-to-fine

Цель: обучать Global не как точный nucleotide detector, а как
**иерархический coarse detector**.

Global должна отвечать на четыре вопроса одновременно:

```text
есть ли TSS в ±256 bp?
есть ли TSS в ±128 bp?
есть ли TSS в ±64 bp?
есть ли TSS в ±32 bp?
```

Архитектура:

```text
9 216 bp DNA
    ↓
BPE-200
    ↓
Global dilated CNN
    ↓
shared global features [32]
    ↓
 ┌────────┬────────┬───────┬───────┐
 ±256     ±128     ±64     ±32
 head     head     head     head
```

Каждый head — `1×1 Conv` в token space, затем prediction
gather'ится обратно на base-resolution.

Важно: это всё ещё одна Global-сеть. Heads только заставляют
один shared encoder одновременно учить грубую и более точную локализацию.

## Target для head радиуса R

Для каждой истинной положительной позиции:

```text
exact TSS = 1
```

строим расширенную область:

```text
[pos-R, pos+R] = 1
```

То есть `R=256` учит сеть находить район шириной до 513 bp,
а `R=32` уже требует гораздо более точной локализации.

Чтобы не создавать ложные negatives у краёв tile, для head `R`
не считаем loss на первых/последних `R` bp центрального target.

## Что проверяем на full holdout

Для каждого head:

1. `Exact AP` — насколько coarse head всё равно сохраняет
   точную информацию о nucleotide.
2. `Coarse AP` — AP против собственного dilated target.
3. `Peak alignment` — где максимум head относительно реального TSS:
   `±4 / ±8 / ±16 / ±32 / ±64 / ±128 / ±256 bp`.

Главная цель этого эксперимента:
поднять `hit within ±256 bp` существенно выше baseline Global-only,
который ранее дал ~0.772.

Фиксировано:

- BPE = `200`;
- тот же BPE artifact/cache;
- global input = `9216 bp`;
- тот же Global encoder;
- seed = `42`;
- batch size = `4`;
- `10 000` steps;
- полный holdout;
- Local CNN отсутствует.


In [1]:
from pathlib import Path
from collections import OrderedDict
import gc
import hashlib
import math
import re
import sys
import time
import warnings

from IPython.display import display
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_directml

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    dml_weighted_profile_bce_with_logits,
)
from ml_project.epic_data import load_split

print("Project:", PROJECT_ROOT)


Project: D:\Projects\EPIC\EPIC


In [2]:
device = torch_directml.device()
print("DirectML device:", device)
print("GPU:", torch_directml.device_name(0))


DirectML device: privateuseone:0
GPU: AMD Radeon RX 6750 GRE 12GB 


In [3]:
split = load_split(PROJECT_ROOT)
sequences, fasta_alphabet = load_baseline_sequences(split)

train_contigs = tuple(sorted(set(split.intervals("train")["contig"].astype(str))))
validation_contigs = tuple(sorted(set(split.intervals("validation")["contig"].astype(str))))
assert set(train_contigs).isdisjoint(validation_contigs)

profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)

GLOBAL_FLANK = 4096
TARGET_LENGTH = profile_config.target_length
LOCAL_FLANK = profile_config.context_flank
GLOBAL_LENGTH = GLOBAL_FLANK + TARGET_LENGTH + GLOBAL_FLANK
LOCAL_LENGTH = LOCAL_FLANK + TARGET_LENGTH + LOCAL_FLANK
LOCAL_START_IN_GLOBAL = GLOBAL_FLANK - LOCAL_FLANK
LOCAL_END_IN_GLOBAL = LOCAL_START_IN_GLOBAL + LOCAL_LENGTH

train_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)
validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=profile_config,
    seed=42,
)

cache_contigs = tuple(dict.fromkeys((*train_contigs, *validation_contigs)))

assert GLOBAL_LENGTH == 9216
assert LOCAL_LENGTH == 1284
assert LOCAL_END_IN_GLOBAL - LOCAL_START_IN_GLOBAL == LOCAL_LENGTH

display(split.summary)
print("Train contigs:", len(train_contigs))
print("Validation contigs:", len(validation_contigs))
print("Cache contigs:", len(cache_contigs))
print("Local / Global:", LOCAL_LENGTH, "/", GLOBAL_LENGTH)


,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


Train contigs: 9
Validation contigs: 3
Cache contigs: 12
Local / Global: 1284 / 9216


In [4]:
# ============================================================
# BPE-200 CONFIGURATION
# ============================================================

BPE_BASE_MERGES = 100
BPE_MERGES = 200

BPE_BASE_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / f"bpe_softmask_{BPE_BASE_MERGES}.pt"
)

BPE_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / f"bpe_softmask_{BPE_MERGES}.pt"
)

BUILD_BPE_IF_MISSING = True

print("Base artifact:", BPE_BASE_PATH)
print("Target artifact:", BPE_PATH)
print(f"BPE-{BPE_MERGES} exists:", BPE_PATH.is_file())

if BPE_MERGES >= 251:
    raise ValueError(
        "Этот notebook использует uint8 token ids; "
        "для >=251 merges нужен более широкий dtype."
    )


Base artifact: D:\Projects\EPIC\EPIC\artifacts\bpe_softmask_100.pt
Target artifact: D:\Projects\EPIC\EPIC\artifacts\bpe_softmask_200.pt
BPE-200 exists: True


In [5]:
# ============================================================
# ONE-TIME STEP: EXTEND BPE-100 -> BPE-200
# ============================================================
# Merge rules are learned ONLY from train contigs.
# Validation sequence is never used to select BPE rules.


def _norm_rules(history):
    return [tuple(int(v) for v in rule) for rule in history]


def _merge_tokens_cpu(
    tokens: torch.Tensor,
    left: int,
    right: int,
    new_token: int,
):
    """Apply one BPE rule exactly left-to-right on a 1D CPU tensor."""

    if tokens.device.type != "cpu":
        raise ValueError("_merge_tokens_cpu expects a CPU tensor")

    if tokens.numel() < 2:
        return tokens

    matches = (
        (tokens[:-1] == left)
        & (tokens[1:] == right)
    )

    starts = torch.nonzero(
        matches,
        as_tuple=False,
    ).flatten()

    if starts.numel() == 0:
        return tokens

    # Overlapping candidates are possible only for left == right.
    # Keep every second candidate inside each consecutive run.
    if left == right and starts.numel() > 1:
        new_run = torch.cat((
            torch.tensor(
                [True],
                dtype=torch.bool,
            ),
            starts[1:] != starts[:-1] + 1,
        ))

        run_ids = (
            torch.cumsum(
                new_run.to(torch.int64),
                dim=0,
            )
            - 1
        )

        run_starts = torch.nonzero(
            new_run,
            as_tuple=False,
        ).flatten()

        offsets = (
            torch.arange(
                starts.numel(),
                dtype=torch.int64,
            )
            - run_starts[run_ids]
        )

        starts = starts[
            offsets.remainder(2) == 0
        ]

    out = tokens.clone()
    out[starts] = new_token

    remove = torch.zeros(
        tokens.numel(),
        dtype=torch.bool,
    )
    remove[starts + 1] = True

    return out[~remove].contiguous()


def _apply_rules_cpu(
    tokens: torch.Tensor,
    rules,
):
    for left, right, new_token in rules:
        tokens = _merge_tokens_cpu(
            tokens,
            left,
            right,
            new_token,
        )
    return tokens


def build_bpe_target_from_base():

    if not BPE_BASE_PATH.is_file():
        raise FileNotFoundError(
            f"Нужен готовый BPE-{BPE_BASE_MERGES}: "
            f"{BPE_BASE_PATH}"
        )

    base = torch.load(
        BPE_BASE_PATH,
        map_location="cpu",
        weights_only=False,
    )

    rules = _norm_rules(
        base["merge_history"]
    )

    if len(rules) != BPE_BASE_MERGES:
        raise ValueError(
            f"Expected BPE-{BPE_BASE_MERGES}, "
            f"got {len(rules)} merges"
        )

    token_text = {
        int(k): str(v)
        for k, v in base["token_text"].items()
    }

    token_length = {
        int(k): int(v)
        for k, v in base["token_length"].items()
    }

    # Replay the existing BPE-100 once on TRAIN contigs.
    # These compact uint8 streams are then used to learn merges 101..200.
    streams = {}
    total_bp = 0
    started = time.perf_counter()

    for i, contig in enumerate(
        train_contigs,
        start=1,
    ):
        raw = torch.as_tensor(
            sequences[contig],
            dtype=torch.uint8,
        )

        tokens = torch.where(
            raw < 8,
            torch.remainder(raw, 4),
            torch.full_like(raw, 4),
        ).to(torch.uint8)

        tokens = _apply_rules_cpu(
            tokens,
            rules,
        ).to(torch.uint8).contiguous()

        streams[contig] = tokens
        total_bp += raw.numel()

        print(
            f"BPE-{BPE_BASE_MERGES} replay "
            f"{i:>3}/{len(train_contigs)} "
            f"{contig}: {tokens.numel():,} tokens"
        )

    PAIR_BASE = 256
    PAIR_SPACE = PAIR_BASE * PAIR_BASE

    while len(rules) < BPE_MERGES:

        counts = torch.zeros(
            PAIR_SPACE,
            dtype=torch.int64,
        )

        for tokens in streams.values():

            if tokens.numel() < 2:
                continue

            pair_code = (
                tokens[:-1].to(torch.int64)
                * PAIR_BASE
                + tokens[1:].to(torch.int64)
            )

            counts += torch.bincount(
                pair_code,
                minlength=PAIR_SPACE,
            )

        best = int(
            torch.argmax(counts)
        )

        best_count = int(
            counts[best]
        )

        left, right = divmod(
            best,
            PAIR_BASE,
        )

        new_token = 5 + len(rules)

        if best_count <= 0:
            raise RuntimeError(
                "No pair left to merge"
            )

        if new_token >= 256:
            raise RuntimeError(
                "uint8 token storage is no longer sufficient"
            )

        for contig in streams:
            streams[contig] = _merge_tokens_cpu(
                streams[contig],
                left,
                right,
                new_token,
            ).to(torch.uint8).contiguous()

        rules.append(
            (left, right, new_token)
        )

        token_text[new_token] = (
            token_text[left]
            + token_text[right]
        )

        token_length[new_token] = (
            token_length[left]
            + token_length[right]
        )

        total_tokens = sum(
            t.numel()
            for t in streams.values()
        )

        print(
            f"merge={len(rules):3d} "
            f"pair=({left},{right}) -> {new_token} "
            f"count={best_count:,} "
            f"compression={total_bp / total_tokens:.4f}x"
        )

    artifact = {
        "merge_history": rules,
        "token_text": token_text,
        "token_length": token_length,
        "vocab_size": 5 + len(rules),
        "num_merges": len(rules),
        "train_bp": int(total_bp),
        "source_artifact": str(BPE_BASE_PATH),
        "case_independent": True,
        "softmask_mode": "per-occurrence lowercase_count",
    }

    torch.save(
        artifact,
        BPE_PATH,
    )

    print()
    print("Saved:", BPE_PATH)
    print(
        "Minutes:",
        round(
            (time.perf_counter() - started) / 60,
            2,
        ),
    )


if not BPE_PATH.is_file():

    if BUILD_BPE_IF_MISSING:
        build_bpe_target_from_base()
    else:
        raise FileNotFoundError(
            BPE_PATH
        )

else:
    print(
        f"Using existing BPE-{BPE_MERGES} artifact"
    )


Using existing BPE-200 artifact


In [6]:
# ============================================================
# LOAD BPE-200 + ARTIFACT INTEGRITY
# ============================================================

bpe_artifact = torch.load(
    BPE_PATH,
    map_location="cpu",
    weights_only=False,
)

merge_history = [
    tuple(int(v) for v in rule)
    for rule in bpe_artifact["merge_history"]
]

token_text = {
    int(k): str(v)
    for k, v in bpe_artifact["token_text"].items()
}

token_length = {
    int(k): int(v)
    for k, v in bpe_artifact["token_length"].items()
}

vocab_size = int(
    bpe_artifact["vocab_size"]
)

assert len(merge_history) == BPE_MERGES
assert vocab_size == 5 + BPE_MERGES
assert set(range(vocab_size)).issubset(token_length)

# BPE-200 must be a strict continuation of the exact BPE-100 artifact.
base_artifact = torch.load(
    BPE_BASE_PATH,
    map_location="cpu",
    weights_only=False,
)

base_rules = [
    tuple(int(v) for v in rule)
    for rule in base_artifact["merge_history"]
]

assert len(base_rules) == BPE_BASE_MERGES
assert merge_history[:BPE_BASE_MERGES] == base_rules

for merge_index, (
    left,
    right,
    new_token,
) in enumerate(merge_history):

    assert new_token == 5 + merge_index

    assert (
        token_length[new_token]
        == token_length[left]
        + token_length[right]
    )

    assert (
        token_text[new_token]
        == token_text[left]
        + token_text[right]
    )

print(
    f"BPE-{BPE_MERGES} artifact integrity: OK"
)

PAD_TOKEN_ID = vocab_size
N_TOKEN_ID = 4

MAX_TOKEN_LENGTH = max(
    token_length.values()
)

if MAX_TOKEN_LENGTH >= 256:
    raise ValueError(
        "uint8 lowercase_count requires "
        "max token length < 256 bp"
    )

TOKEN_LENGTH_LOOKUP = torch.zeros(
    vocab_size,
    dtype=torch.int16,
)

for token_id, length in token_length.items():
    TOKEN_LENGTH_LOOKUP[token_id] = length

MERGE_HASH = hashlib.sha1(
    repr(merge_history).encode()
).hexdigest()[:12]

CACHE_FORMAT_VERSION = 2
CACHE_ENGINE = "torch_cpu_exact_left_to_right"

# Separate cache: BPE-100 cache can never be reused accidentally.
CACHE_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / (
        f"bpe{BPE_MERGES}_pretokenized_"
        f"v{CACHE_FORMAT_VERSION}_{MERGE_HASH}"
    )
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("BPE merges:", len(merge_history))
print("Vocab:", vocab_size)
print("Max token length:", MAX_TOKEN_LENGTH)
print("Merge hash:", MERGE_HASH)
print("Cache engine:", CACHE_ENGINE)
print("Cache:", CACHE_DIR)


BPE-200 artifact integrity: OK
BPE merges: 200
Vocab: 205
Max token length: 8
Merge hash: 03d3d5e77ea4
Cache engine: torch_cpu_exact_left_to_right
Cache: D:\Projects\EPIC\EPIC\artifacts\bpe200_pretokenized_v2_03d3d5e77ea4


In [7]:
# ============================================================
# EXACT ONE-TIME CPU PRETOKENIZATION — BPE-200
# ============================================================
# CPU is intentional here. The old full-contig DirectML path produced
# invalid no-merge caches on long contigs. Training itself remains GPU-based.

INDEX_CHUNK_TOKENS = 1_000_000


def reverse_complement_storage_codes_cpu(raw: torch.Tensor) -> torch.Tensor:
    if raw.device.type != "cpu":
        raise ValueError("reverse_complement_storage_codes_cpu expects CPU")
    x = raw.to(torch.int16)
    is_base = x < 8
    case = ((x >= 4) & (x < 8)).to(torch.int16) * 4
    rc = 3 - torch.remainder(x, 4) + case
    rc = torch.where(is_base, rc, torch.full_like(rc, 8))
    return rc.flip(0).to(torch.uint8).contiguous()


def _merge_tokens_and_lower_cpu(
    tokens: torch.Tensor,
    lower: torch.Tensor,
    left: int,
    right: int,
    new_token: int,
):
    """Exact left-to-right merge preserving per-token lowercase_count."""
    if tokens.device.type != "cpu" or lower.device.type != "cpu":
        raise ValueError("CPU tensors expected")
    if tokens.numel() != lower.numel():
        raise ValueError("tokens/lower length mismatch")
    if tokens.numel() < 2:
        return tokens, lower

    matches = (tokens[:-1] == left) & (tokens[1:] == right)
    starts = torch.nonzero(matches, as_tuple=False).flatten()
    if starts.numel() == 0:
        return tokens, lower

    if left == right and starts.numel() > 1:
        new_run = torch.cat((
            torch.tensor([True], dtype=torch.bool),
            starts[1:] != starts[:-1] + 1,
        ))
        run_ids = torch.cumsum(new_run.to(torch.int64), dim=0) - 1
        run_starts = torch.nonzero(new_run, as_tuple=False).flatten()
        offsets = torch.arange(starts.numel(), dtype=torch.int64) - run_starts[run_ids]
        starts = starts[offsets.remainder(2) == 0]

    out_tokens = tokens.clone()
    out_lower = lower.clone()
    out_tokens[starts] = new_token
    out_lower[starts] = lower[starts] + lower[starts + 1]

    remove = torch.zeros(tokens.numel(), dtype=torch.bool)
    remove[starts + 1] = True
    keep = ~remove

    return (
        out_tokens[keep].contiguous(),
        out_lower[keep].contiguous(),
    )


def apply_bpe_cpu(raw_cpu: torch.Tensor, *, reverse_complement: bool):
    raw = raw_cpu.to(dtype=torch.uint8, device="cpu").contiguous()
    if reverse_complement:
        raw = reverse_complement_storage_codes_cpu(raw)

    tokens = torch.where(
        raw < 8,
        torch.remainder(raw, 4),
        torch.full_like(raw, 4),
    ).to(torch.uint8).contiguous()
    lower = ((raw >= 4) & (raw < 8)).to(torch.uint8).contiguous()

    for left, right, new_token in merge_history:
        tokens, lower = _merge_tokens_and_lower_cpu(
            tokens, lower, left, right, new_token
        )

    return tokens, lower


def safe_name(contig: str):
    stem = re.sub(r"[^A-Za-z0-9._-]+", "_", contig)[:80]
    suffix = hashlib.sha1(contig.encode()).hexdigest()[:10]
    return f"{stem}_{suffix}"


def cache_path(contig: str, orientation: str):
    return CACHE_DIR / f"{safe_name(contig)}.{orientation}.pt"


def _build_token_ends(token_ids: torch.Tensor):
    n_tokens = token_ids.numel()
    ends = torch.empty(n_tokens, dtype=torch.int32)
    running_bp = 0

    for start in range(0, n_tokens, INDEX_CHUNK_TOKENS):
        stop = min(start + INDEX_CHUNK_TOKENS, n_tokens)
        lengths = TOKEN_LENGTH_LOOKUP[token_ids[start:stop].long()].to(torch.int32)
        chunk_ends = torch.cumsum(lengths, dim=0) + running_bp
        ends[start:stop] = chunk_ends
        running_bp = int(chunk_ends[-1])

    return ends, running_bp


def cache_is_compatible(path: Path, contig: str, orientation: str):
    if not path.is_file():
        return False
    try:
        p = torch.load(path, map_location="cpu", weights_only=False, mmap=True)
        return (
            int(p.get("cache_format_version", -1)) == CACHE_FORMAT_VERSION
            and p.get("cache_engine") == CACHE_ENGINE
            and p.get("merge_hash") == MERGE_HASH
            and int(p.get("bpe_merges", -1)) == BPE_MERGES
            and p.get("contig") == contig
            and p.get("orientation") == orientation
            and "token_ids" in p
            and "lowercase_count" in p
            and "token_ends" in p
        )
    except Exception:
        return False


def pretokenize_contig_cpu(contig: str, orientation: str, overwrite=False):
    if orientation not in {"plus", "rc"}:
        raise ValueError(orientation)

    path = cache_path(contig, orientation)
    if not overwrite and cache_is_compatible(path, contig, orientation):
        print(f"{contig} [{orientation}] cached")
        return path

    raw = torch.as_tensor(sequences[contig], dtype=torch.uint8).contiguous()
    lowercase_bp = int(((raw >= 4) & (raw < 8)).sum())

    ids, lower = apply_bpe_cpu(
        raw,
        reverse_complement=(orientation == "rc"),
    )
    ends, reconstructed_bp = _build_token_ends(ids)

    assert reconstructed_bp == raw.numel()
    assert ids.numel() == lower.numel() == ends.numel()
    assert int(lower.to(torch.int64).sum()) == lowercase_bp
    assert int(ids.max()) < vocab_size if ids.numel() else True

    payload = {
        "cache_format_version": CACHE_FORMAT_VERSION,
        "cache_engine": CACHE_ENGINE,
        "contig": contig,
        "orientation": orientation,
        "length_bp": int(raw.numel()),
        "lowercase_bp": lowercase_bp,
        "token_ids": ids,
        "lowercase_count": lower,
        "token_ends": ends,
        "merge_hash": MERGE_HASH,
        "bpe_merges": BPE_MERGES,
        "vocab_size": vocab_size,
    }
    torch.save(payload, path)

    print(
        f"{contig} [{orientation}] bp={raw.numel():,} "
        f"tokens={ids.numel():,} compression={raw.numel() / ids.numel():.3f}x"
    )

    del raw, ids, lower, ends, payload
    gc.collect()
    return path


In [8]:
# ============================================================
# CPU-ONLY BPE SEMANTIC SMOKE TEST
# ============================================================
# Full-contig BPE is intentionally CPU-only.
# DirectML is not touched here.

smoke_contig = train_contigs[0]

smoke_raw = torch.as_tensor(
    sequences[smoke_contig][:GLOBAL_LENGTH],
    dtype=torch.uint8,
).contiguous()

if smoke_raw.numel() < GLOBAL_LENGTH:
    smoke_raw = torch.cat((
        smoke_raw,
        torch.full(
            (GLOBAL_LENGTH - smoke_raw.numel(),),
            8,
            dtype=torch.uint8,
        ),
    ))

for rc in (False, True):

    ids, lower = apply_bpe_cpu(
        smoke_raw,
        reverse_complement=rc,
    )

    lengths = TOKEN_LENGTH_LOOKUP[
        ids.long()
    ].to(torch.int32)

    assert ids.dtype == torch.uint8
    assert lower.dtype == torch.uint8
    assert ids.numel() == lower.numel()
    assert int(lengths.sum()) == GLOBAL_LENGTH

    assert torch.all(
        lower.to(torch.int16)
        <= lengths.to(torch.int16)
    )

    source = (
        reverse_complement_storage_codes_cpu(smoke_raw)
        if rc
        else smoke_raw
    )

    expected_lowercase = int(
        (
            (source >= 4)
            & (source < 8)
        ).sum()
    )

    assert (
        int(lower.to(torch.int64).sum())
        == expected_lowercase
    )

    print(
        f"{'RC' if rc else 'PLUS'}: "
        f"tokens={ids.numel():,} "
        f"compression="
        f"{GLOBAL_LENGTH / ids.numel():.3f}x"
    )

print(
    f"BPE-{BPE_MERGES} CPU semantic check: OK"
)


PLUS: tokens=2,795 compression=3.297x
RC: tokens=2,795 compression=3.297x
BPE-200 CPU semantic check: OK


In [9]:
# ============================================================
# BUILD EXACT BPE-200 PRETOKENIZED CACHE — ONE TIME, RESUME-SAFE
# ============================================================
# Old DirectML cache is intentionally ignored because CACHE_DIR changed.

OVERWRITE_CACHE = False

started = time.perf_counter()
for i, contig in enumerate(cache_contigs, 1):
    print(f"\n[{i}/{len(cache_contigs)}] {contig}")
    pretokenize_contig_cpu(contig, "plus", OVERWRITE_CACHE)
    pretokenize_contig_cpu(contig, "rc", OVERWRITE_CACHE)

print()
print("Exact cache ready. Minutes:", round((time.perf_counter() - started) / 60, 2))



[1/12] NC_064035.1
NC_064035.1 [plus] cached
NC_064035.1 [rc] cached

[2/12] NC_064036.1
NC_064036.1 [plus] cached
NC_064036.1 [rc] cached

[3/12] NC_064038.1
NC_064038.1 [plus] cached
NC_064038.1 [rc] cached

[4/12] NC_064039.1
NC_064039.1 [plus] cached
NC_064039.1 [rc] cached

[5/12] NC_064040.1
NC_064040.1 [plus] cached
NC_064040.1 [rc] cached

[6/12] NC_064043.1
NC_064043.1 [plus] cached
NC_064043.1 [rc] cached

[7/12] NC_064045.1
NC_064045.1 [plus] cached
NC_064045.1 [rc] cached

[8/12] NC_064047.1
NC_064047.1 [plus] cached
NC_064047.1 [rc] cached

[9/12] NC_064048.1
NC_064048.1 [plus] cached
NC_064048.1 [rc] cached

[10/12] NC_064034.1
NC_064034.1 [plus] cached
NC_064034.1 [rc] cached

[11/12] NC_064041.1
NC_064041.1 [plus] cached
NC_064041.1 [rc] cached

[12/12] NC_064042.1
NC_064042.1 [plus] cached
NC_064042.1 [rc] cached

Exact cache ready. Minutes: 0.0


In [10]:
# ============================================================
# FULL CACHE AUDIT
# ============================================================
# Verifies metadata, bp reconstruction, lowercase counts and boundary index.

AUDIT_CHUNK_TOKENS = 1_000_000


def audit_cache_file(contig: str, orientation: str):
    path = cache_path(contig, orientation)
    if not cache_is_compatible(path, contig, orientation):
        raise RuntimeError(f"Incompatible cache: {path}")

    p = torch.load(path, map_location="cpu", weights_only=False, mmap=True)
    ids = p["token_ids"]
    lower = p["lowercase_count"]
    ends = p["token_ends"]

    raw = torch.as_tensor(sequences[contig], dtype=torch.uint8)
    expected_bp = raw.numel()
    expected_lower = int(((raw >= 4) & (raw < 8)).sum())

    assert int(p["length_bp"]) == expected_bp
    assert ids.dtype == torch.uint8
    assert lower.dtype == torch.uint8
    assert ends.dtype == torch.int32
    assert ids.numel() == lower.numel() == ends.numel()
    assert ids.numel() > 0
    assert int(ids.max()) < vocab_size

    running_bp = 0
    lower_total = 0

    for start in range(0, ids.numel(), AUDIT_CHUNK_TOKENS):
        stop = min(start + AUDIT_CHUNK_TOKENS, ids.numel())
        chunk_ids = ids[start:stop]
        chunk_lower = lower[start:stop]
        chunk_ends = ends[start:stop]
        lengths = TOKEN_LENGTH_LOOKUP[chunk_ids.long()].to(torch.int32)
        expected_ends = torch.cumsum(lengths, dim=0) + running_bp

        assert torch.equal(chunk_ends, expected_ends)
        assert torch.all(chunk_lower.to(torch.int16) <= lengths.to(torch.int16))

        running_bp = int(expected_ends[-1])
        lower_total += int(chunk_lower.to(torch.int64).sum())

    assert running_bp == expected_bp
    assert lower_total == expected_lower == int(p["lowercase_bp"])

    return {
        "contig": contig,
        "orientation": orientation,
        "bp": expected_bp,
        "tokens": ids.numel(),
        "compression": expected_bp / ids.numel(),
        "lowercase_bp": lower_total,
    }


audit_rows = []
started = time.perf_counter()

for i, contig in enumerate(cache_contigs, 1):
    print(f"[{i}/{len(cache_contigs)}] {contig}")
    for orientation in ("plus", "rc"):
        row = audit_cache_file(contig, orientation)
        audit_rows.append(row)
        print(
            f"  {orientation:4s} tokens={row['tokens']:,} "
            f"compression={row['compression']:.3f}x"
        )

cache_audit = pd.DataFrame(audit_rows)
display(cache_audit)
print("Audit minutes:", round((time.perf_counter() - started) / 60, 2))
print("CACHE AUDIT: OK")


[1/12] NC_064035.1
  plus tokens=6,575,767 compression=3.350x
  rc   tokens=6,572,480 compression=3.352x
[2/12] NC_064036.1
  plus tokens=6,214,416 compression=3.346x
  rc   tokens=6,211,319 compression=3.348x
[3/12] NC_064038.1
  plus tokens=5,669,670 compression=3.351x
  rc   tokens=5,674,624 compression=3.348x
[4/12] NC_064039.1
  plus tokens=5,406,734 compression=3.347x
  rc   tokens=5,407,155 compression=3.347x
[5/12] NC_064040.1
  plus tokens=5,323,448 compression=3.356x
  rc   tokens=5,321,541 compression=3.357x
[6/12] NC_064043.1
  plus tokens=4,943,267 compression=3.357x
  rc   tokens=4,953,485 compression=3.350x
[7/12] NC_064045.1
  plus tokens=4,889,799 compression=3.352x
  rc   tokens=4,891,140 compression=3.351x
[8/12] NC_064047.1
  plus tokens=4,596,057 compression=3.345x
  rc   tokens=4,593,831 compression=3.346x
[9/12] NC_064048.1
  plus tokens=4,074,317 compression=3.367x
  rc   tokens=4,073,645 compression=3.367x
[10/12] NC_064034.1
  plus tokens=6,627,064 compression

,contig,orientation,bp,tokens,compression,lowercase_bp
0,NC_064035.1,plus,22031953,6575767,3.350477,5303374
1,NC_064035.1,rc,22031953,6572480,3.352152,5303374
2,NC_064036.1,plus,20793841,6214416,3.346065,5356286
3,NC_064036.1,rc,20793841,6211319,3.347734,5356286
4,NC_064038.1,plus,19000648,5669670,3.351279,7404189
5,NC_064038.1,rc,19000648,5674624,3.348354,7404189
6,NC_064039.1,plus,18096078,5406734,3.346952,4604642
7,NC_064039.1,rc,18096078,5407155,3.346691,4604642
8,NC_064040.1,plus,17865913,5323448,3.356079,5523134
9,NC_064040.1,rc,17865913,5321541,3.357282,5523134


Audit minutes: 0.02
CACHE AUDIT: OK


In [11]:
# ============================================================
# LAZY MMAP CACHE
# ============================================================
# These are mmap handles, not full copies of all tensors in RAM.
# Keeping all 24 orientation files mapped avoids repeated torch.load in training.

CACHE_MAX_ITEMS = max(32, 2 * len(cache_contigs))
_CACHE_LRU = OrderedDict()


def load_cached_lazy(contig: str, orientation: str):
    key = (contig, orientation)

    if key in _CACHE_LRU:
        item = _CACHE_LRU.pop(key)
        _CACHE_LRU[key] = item
        return item

    path = cache_path(contig, orientation)
    if not cache_is_compatible(path, contig, orientation):
        raise RuntimeError(f"Missing/incompatible cache: {path}")

    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
        mmap=True,
    )

    item = {
        "payload": payload,  # keeps mmap storage alive
        "length_bp": int(payload["length_bp"]),
        "ids": payload["token_ids"],
        "lower": payload["lowercase_count"],
        "ends": payload["token_ends"],
    }

    _CACHE_LRU[key] = item
    while len(_CACHE_LRU) > CACHE_MAX_ITEMS:
        _CACHE_LRU.popitem(last=False)
        gc.collect()

    return item


print("Lazy mmap cache ready.")
print("Mapped files now:", len(_CACHE_LRU))
print("No full-contig lengths/cumsum tensors are created in RAM.")


Lazy mmap cache ready.
Mapped files now: 0
No full-contig lengths/cumsum tensors are created in RAM.


In [12]:
# ============================================================
# FAST WINDOW SLICING — NO BPE MERGES HERE
# ============================================================


def _search_token_index(ends: torch.Tensor, bp_coordinate: int):
    query = torch.tensor(bp_coordinate, dtype=ends.dtype)
    return int(torch.searchsorted(ends, query, right=True))


def slice_cached(contig, genomic_start, genomic_end, *, rc=False):
    orientation = "rc" if rc else "plus"
    item = load_cached_lazy(contig, orientation)
    L = item["length_bp"]

    if rc:
        start, end = L - genomic_end, L - genomic_start
    else:
        start, end = genomic_start, genomic_end

    if end - start != GLOBAL_LENGTH:
        raise AssertionError((start, end, GLOBAL_LENGTH))

    left_pad = max(0, -start)
    right_pad = max(0, end - L)
    a, b = max(0, start), min(L, end)

    ids_all = item["ids"]
    lower_all = item["lower"]
    ends_all = item["ends"]

    if b > a:
        first = _search_token_index(ends_all, a)
        last = _search_token_index(ends_all, b - 1)
        stop = last + 1

        core_ids = ids_all[first:stop]
        core_lower = lower_all[first:stop]
        core_lengths = TOKEN_LENGTH_LOOKUP[core_ids.long()].to(torch.int16)
        core_ends_abs = ends_all[first:stop]

        # Search only the small token slice, not the full 20M-element index.
        coords = torch.arange(a, b, dtype=core_ends_abs.dtype)
        core_map = (
            torch.searchsorted(core_ends_abs, coords, right=True)
            + left_pad
        ).to(torch.long)
    else:
        core_ids = torch.empty(0, dtype=torch.uint8)
        core_lower = torch.empty(0, dtype=torch.uint8)
        core_lengths = torch.empty(0, dtype=torch.int16)
        core_map = torch.empty(0, dtype=torch.long)

    # N is intentionally never BPE-merged, so 1 padded bp = 1 N token.
    left_ids = torch.full((left_pad,), N_TOKEN_ID, dtype=torch.uint8)
    left_lower = torch.zeros(left_pad, dtype=torch.uint8)
    left_lengths = torch.ones(left_pad, dtype=torch.int16)
    left_map = torch.arange(left_pad, dtype=torch.long)

    right_ids = torch.full((right_pad,), N_TOKEN_ID, dtype=torch.uint8)
    right_lower = torch.zeros(right_pad, dtype=torch.uint8)
    right_lengths = torch.ones(right_pad, dtype=torch.int16)
    right_map = (
        left_pad
        + core_ids.numel()
        + torch.arange(right_pad, dtype=torch.long)
    )

    ids = torch.cat((left_ids, core_ids, right_ids)).contiguous()
    lower = torch.cat((left_lower, core_lower, right_lower)).contiguous()
    lengths = torch.cat((left_lengths, core_lengths, right_lengths)).contiguous()
    bp_to_token = torch.cat((left_map, core_map, right_map)).contiguous()

    assert bp_to_token.numel() == GLOBAL_LENGTH
    assert ids.numel() == lengths.numel() == lower.numel()
    assert bp_to_token.min() >= 0
    assert bp_to_token.max() < ids.numel()

    return {
        "ids": ids,
        "lengths": lengths,
        "softmask": lower.float() / lengths.float(),
        "bp_to_token": bp_to_token,
    }


def build_cached_both_batch(tiles):
    items = []

    # Ordering intentionally matches batch.x_both_strands:
    # all PLUS rows first, then all RC rows.
    for rc in (False, True):
        for i in range(len(tiles)):
            tile = tiles.iloc[i]
            item = slice_cached(
                str(tile["contig"]),
                int(tile["target_start"]) - GLOBAL_FLANK,
                int(tile["target_end"]) + GLOBAL_FLANK,
                rc=rc,
            )
            item["local_map"] = item["bp_to_token"][
                LOCAL_START_IN_GLOBAL:LOCAL_END_IN_GLOBAL
            ]
            assert item["local_map"].numel() == LOCAL_LENGTH
            items.append(item)

    T = max(x["ids"].numel() for x in items)
    N = len(items)

    token_ids = torch.full((N, T), PAD_TOKEN_ID, dtype=torch.long)
    token_lengths = torch.zeros((N, T), dtype=torch.float32)
    softmask = torch.zeros((N, T), dtype=torch.float32)
    token_mask = torch.zeros((N, T), dtype=torch.float32)  # float for DirectML
    local_map = torch.empty((N, LOCAL_LENGTH), dtype=torch.long)

    for row, x in enumerate(items):
        n = x["ids"].numel()
        token_ids[row, :n] = x["ids"].long()
        token_lengths[row, :n] = x["lengths"].float()
        softmask[row, :n] = x["softmask"]
        token_mask[row, :n] = 1.0
        local_map[row] = x["local_map"]

    return {
        "token_ids": token_ids.contiguous(),
        "token_lengths": token_lengths.contiguous(),
        "softmask": softmask.contiguous(),
        "token_mask": token_mask.contiguous(),
        "local_map": local_map.contiguous(),
    }


# Real batch smoke-test.
smoke_batch = train_generator.sample_batch(batch_size=4)
t0 = time.perf_counter()
smoke_global = build_cached_both_batch(smoke_batch.tiles)
cache_build_ms = (time.perf_counter() - t0) * 1000

print("Cached batch build ms:", round(cache_build_ms, 2))
print("Token batch:", smoke_global["token_ids"].shape)
print("Mapped files:", len(_CACHE_LRU))
assert smoke_global["token_ids"].shape[0] == 8
assert smoke_global["local_map"].shape == (8, LOCAL_LENGTH)
print("Cached slicing smoke-test: OK")


Cached batch build ms: 9.3
Token batch: torch.Size([8, 2825])
Mapped files: 6
Cached slicing smoke-test: OK


In [13]:
# ============================================================
# MODEL — GLOBAL BPE-200 MULTI-SCALE COARSE HEADS
# ============================================================

GLOBAL_DILATIONS = (
    1, 2, 4, 8, 16,
    32, 64, 128, 256, 512,
)

GLOBAL_TOKEN_RF = (
    1
    + sum(
        4 * d
        for d in GLOBAL_DILATIONS
    )
)

assert GLOBAL_TOKEN_RF == 4093

COARSE_RADII = (
    256,
    128,
    64,
    32,
)


class GlobalResidualBlock(nn.Module):
    def __init__(self, channels, dilation):
        super().__init__()

        self.conv1 = nn.Conv1d(
            channels,
            channels,
            3,
            padding=dilation,
            dilation=dilation,
        )

        self.conv2 = nn.Conv1d(
            channels,
            channels,
            3,
            padding=dilation,
            dilation=dilation,
        )

        self.act = nn.GELU()

    def forward(self, x, mask):
        m = mask.unsqueeze(1).to(dtype=x.dtype)

        residual = x

        x = (
            self.act(
                self.conv1(x)
            )
            * m
        )

        return (
            self.act(
                self.conv2(x)
                + residual
            )
            * m
        )


class GlobalBPEEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size + 1,
            16,
            padding_idx=PAD_TOKEN_ID,
        )

        self.proj = nn.Sequential(
            nn.Conv1d(
                18,
                32,
                1,
            ),
            nn.GELU(),
        )

        self.blocks = nn.ModuleList(
            GlobalResidualBlock(
                32,
                d,
            )
            for d in GLOBAL_DILATIONS
        )

    def forward(
        self,
        ids,
        lengths,
        softmask,
        mask,
    ):
        emb = self.embedding(ids)

        length_feature = (
            lengths.to(emb.dtype)
            / float(MAX_TOKEN_LENGTH)
        ).unsqueeze(-1)

        soft_feature = (
            softmask.to(emb.dtype)
            .unsqueeze(-1)
        )

        x = torch.cat(
            (
                emb,
                soft_feature,
                length_feature,
            ),
            dim=-1,
        )

        x = (
            x
            * mask.unsqueeze(-1).to(
                dtype=x.dtype
            )
        )

        x = self.proj(
            x.transpose(
                1,
                2,
            ).contiguous()
        )

        x = (
            x
            * mask.unsqueeze(1).to(
                dtype=x.dtype
            )
        )

        for block in self.blocks:
            x = block(
                x,
                mask,
            )

        return x


class GlobalCoarseToFineBPE200Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.global_encoder = GlobalBPEEncoder()

        self.coarse_heads = nn.ModuleDict({
            str(radius): nn.Conv1d(
                32,
                1,
                1,
            )
            for radius in COARSE_RADII
        })

    def forward(
        self,
        ids,
        lengths,
        softmask,
        mask,
        local_map,
    ):
        features = self.global_encoder(
            ids,
            lengths,
            softmask,
            mask,
        )

        gather_index = (
            local_map.unsqueeze(1)
            .contiguous()
        )

        outputs = {}

        for radius in COARSE_RADII:
            token_logits = self.coarse_heads[
                str(radius)
            ](
                features
            )

            bp_logits = torch.gather(
                token_logits,
                2,
                gather_index,
            )

            outputs[
                radius
            ] = bp_logits

        return outputs


def create_model():
    return GlobalCoarseToFineBPE200Model()


def forward_both(
    model,
    global_batch,
    *,
    grad=True,
):
    B2 = global_batch[
        "token_ids"
    ].shape[0]

    if B2 % 2 != 0:
        raise ValueError(
            f"Expected PLUS+RC rows, got {B2}"
        )

    B = B2 // 2

    g = {
        key: value.contiguous().to(device)
        for key, value
        in global_batch.items()
    }

    context = (
        torch.enable_grad()
        if grad
        else torch.no_grad()
    )

    with context:
        raw = model(
            g["token_ids"],
            g["token_lengths"],
            g["softmask"],
            g["token_mask"],
            g["local_map"],
        )

        outputs = {}

        for radius, logits in raw.items():
            logits = logits[
                :,
                :,
                LOCAL_FLANK:-LOCAL_FLANK,
            ]

            outputs[
                radius
            ] = (
                logits[:B],
                logits[B:],
            )

    return outputs


def dilate_binary_target_cpu(
    target,
    radius,
):
    if target.device.type != "cpu":
        raise ValueError(
            "Expected CPU target before DirectML transfer"
        )

    if radius == 0:
        return target.float()

    kernel = (
        2 * radius
        + 1
    )

    return F.max_pool1d(
        target.float(),
        kernel_size=kernel,
        stride=1,
        padding=radius,
    )


def coarse_weight_cpu(
    base_weight,
    radius,
):
    w = (
        base_weight
        .float()
        .clone()
    )

    if radius > 0:
        w[..., :radius] = 0.0
        w[..., -radius:] = 0.0

    return w


model = create_model().to(device)

encoder_parameters = sum(
    p.numel()
    for p in model.global_encoder.parameters()
)

head_parameters = sum(
    p.numel()
    for p in model.coarse_heads.parameters()
)

print(
    "Global token RF:",
    GLOBAL_TOKEN_RF,
    "tokens",
)

print(
    "Global input:",
    GLOBAL_LENGTH,
    "bp",
)

print(
    "Coarse radii:",
    COARSE_RADII,
)

print(
    "Encoder parameters:",
    f"{encoder_parameters:,}",
)

print(
    "Heads parameters:",
    f"{head_parameters:,}",
)

model.eval()

smoke_outputs = forward_both(
    model,
    smoke_global,
    grad=False,
)

for radius in COARSE_RADII:
    pp, mp = smoke_outputs[
        radius
    ]

    print(
        f"R={radius}:",
        pp.shape,
        mp.shape,
    )

    assert pp.shape == (
        4,
        1,
        TARGET_LENGTH,
    )

    assert mp.shape == (
        4,
        1,
        TARGET_LENGTH,
    )

print(
    "GLOBAL coarse-to-fine forward: OK"
)


Global token RF: 4093 tokens
Global input: 9216 bp
Coarse radii: (256, 128, 64, 32)
Encoder parameters: 65,984
Heads parameters: 132
R=256: torch.Size([4, 1, 1024]) torch.Size([4, 1, 1024])
R=128: torch.Size([4, 1, 1024]) torch.Size([4, 1, 1024])
R=64: torch.Size([4, 1, 1024]) torch.Size([4, 1, 1024])
R=32: torch.Size([4, 1, 1024]) torch.Size([4, 1, 1024])
GLOBAL coarse-to-fine forward: OK


In [14]:
# ============================================================
# ONE REAL MULTI-HEAD BACKWARD STEP
# ============================================================

model.train()

optimizer = DirectMLAdam(
    model.parameters(),
    lr=1e-3,
)

optimizer.zero_grad(
    set_to_none=True
)

batch = smoke_batch

plus_exact = (
    batch.plus_target
    .contiguous()
)

minus_exact = (
    batch.minus_target_reverse
    .contiguous()
)

plus_base_w = (
    batch.plus_loss_weight
    .contiguous()
)

minus_base_w = (
    batch.minus_loss_weight_reverse
    .contiguous()
)

with warnings.catch_warnings(
    record=True
) as caught:

    warnings.simplefilter(
        "always"
    )

    outputs = forward_both(
        model,
        smoke_global,
        grad=True,
    )

    head_losses = {}

    for radius in COARSE_RADII:
        pp, mp = outputs[
            radius
        ]

        plus_target_r = (
            dilate_binary_target_cpu(
                plus_exact,
                radius,
            )
            .to(device)
        )

        minus_target_r = (
            dilate_binary_target_cpu(
                minus_exact,
                radius,
            )
            .to(device)
        )

        plus_w_r = (
            coarse_weight_cpu(
                plus_base_w,
                radius,
            )
            .to(device)
        )

        minus_w_r = (
            coarse_weight_cpu(
                minus_base_w,
                radius,
            )
            .to(device)
        )

        head_losses[
            radius
        ] = (
            dml_weighted_profile_bce_with_logits(
                pp,
                mp,
                plus_target_r,
                minus_target_r,
                plus_w_r,
                minus_w_r,
            )
        )

    total_loss = sum(
        head_losses.values()
    ) / len(
        head_losses
    )

    total_loss.backward()

    encoder_grad = sum(
        float(
            p.grad.detach()
            .abs()
            .sum()
            .cpu()
        )
        for p
        in model.global_encoder.parameters()
        if p.grad is not None
    )

    heads_grad = sum(
        float(
            p.grad.detach()
            .abs()
            .sum()
            .cpu()
        )
        for p
        in model.coarse_heads.parameters()
        if p.grad is not None
    )

    optimizer.step()

fallback = [
    str(w.message)
    for w in caught
    if "fall back"
    in str(w.message).lower()
]

for radius in COARSE_RADII:
    print(
        f"R={radius} loss:",
        head_losses[
            radius
        ].item(),
    )

print(
    "Total:",
    total_loss.item(),
)

print(
    "Encoder grad:",
    encoder_grad,
)

print(
    "Heads grad:",
    heads_grad,
)

print(
    "CPU fallback:",
    fallback,
)

assert encoder_grad > 0
assert heads_grad > 0
assert not fallback

print(
    "Coarse-to-fine backward + optimizer step: OK"
)


R=256 loss: 0.3384614586830139
R=128 loss: 0.6061195135116577
R=64 loss: 0.64454585313797
R=32 loss: 0.6668378114700317
Total: 0.5639911890029907
Encoder grad: 0.1146369829075411
Heads grad: 0.01695766660850495
CPU fallback: []
Coarse-to-fine backward + optimizer step: OK


In [15]:
# ============================================================
# 10 000-STEP BPE-200 GLOBAL COARSE-TO-FINE RUN
# ============================================================

STEPS = 10_000
BATCH_SIZE = 4
LOG_EVERY = 100
SEED = 42

RUN_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_EXP017_BPE200_GLOBAL_COARSE_TO_FINE"
    / "run_001"
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_PATH = (
    RUN_DIR
    / f"checkpoint_step_{STEPS:05d}.pt"
)

HISTORY_PATH = (
    RUN_DIR
    / "training_history.csv"
)

torch.manual_seed(SEED)

generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=SEED,
)

model = create_model().to(device)

optimizer = DirectMLAdam(
    model.parameters(),
    lr=1e-3,
)

model.train()

history = []

ema = {
    radius: None
    for radius in COARSE_RADII
}

ema_total = None

seen = 0
prep_seconds = 0.0

started = time.perf_counter()

with warnings.catch_warnings(
    record=True
) as caught:

    warnings.simplefilter(
        "always"
    )

    for step in range(
        1,
        STEPS + 1,
    ):
        t0 = time.perf_counter()

        batch = generator.sample_batch(
            batch_size=BATCH_SIZE
        )

        gb = build_cached_both_batch(
            batch.tiles
        )

        prep_seconds += (
            time.perf_counter()
            - t0
        )

        plus_exact = (
            batch.plus_target
            .contiguous()
        )

        minus_exact = (
            batch.minus_target_reverse
            .contiguous()
        )

        plus_base_w = (
            batch.plus_loss_weight
            .contiguous()
        )

        minus_base_w = (
            batch.minus_loss_weight_reverse
            .contiguous()
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        outputs = forward_both(
            model,
            gb,
            grad=True,
        )

        head_losses = {}

        for radius in COARSE_RADII:
            pp, mp = outputs[
                radius
            ]

            plus_target_r = (
                dilate_binary_target_cpu(
                    plus_exact,
                    radius,
                )
                .to(device)
            )

            minus_target_r = (
                dilate_binary_target_cpu(
                    minus_exact,
                    radius,
                )
                .to(device)
            )

            plus_w_r = (
                coarse_weight_cpu(
                    plus_base_w,
                    radius,
                )
                .to(device)
            )

            minus_w_r = (
                coarse_weight_cpu(
                    minus_base_w,
                    radius,
                )
                .to(device)
            )

            head_losses[
                radius
            ] = (
                dml_weighted_profile_bce_with_logits(
                    pp,
                    mp,
                    plus_target_r,
                    minus_target_r,
                    plus_w_r,
                    minus_w_r,
                )
            )

        loss = sum(
            head_losses.values()
        ) / len(
            head_losses
        )

        value = loss.item()

        if not math.isfinite(value):
            raise FloatingPointError(
                f"Non-finite loss at step {step}"
            )

        loss.backward()
        optimizer.step()

        loss_values = {
            radius: head_losses[
                radius
            ].item()
            for radius in COARSE_RADII
        }

        for radius in COARSE_RADII:
            v = loss_values[
                radius
            ]

            if ema[
                radius
            ] is None:
                ema[
                    radius
                ] = v
            else:
                ema[
                    radius
                ] = (
                    0.95
                    * ema[
                        radius
                    ]
                    + 0.05
                    * v
                )

        if ema_total is None:
            ema_total = value
        else:
            ema_total = (
                0.95
                * ema_total
                + 0.05
                * value
            )

        seen += int(
            batch.mask.sum()
        )

        elapsed = (
            time.perf_counter()
            - started
        )

        row = {
            "step": step,
            "total_loss": value,
            "ema_total": ema_total,
            "positions_per_second": (
                seen / elapsed
            ),
            "prep_fraction": (
                prep_seconds
                / elapsed
            ),
        }

        for radius in COARSE_RADII:
            row[
                f"loss_r{radius}"
            ] = loss_values[
                radius
            ]

            row[
                f"ema_r{radius}"
            ] = ema[
                radius
            ]

        history.append(
            row
        )

        if (
            step == 1
            or step % LOG_EVERY == 0
        ):
            head_text = " ".join(
                f"r{radius}="
                f"{loss_values[radius]:.4f}"
                for radius
                in COARSE_RADII
            )

            ema_text = " ".join(
                f"e{radius}="
                f"{ema[radius]:.4f}"
                for radius
                in COARSE_RADII
            )

            print(
                f"step={step:5d} "
                f"{head_text} "
                f"total={value:.4f} "
                f"{ema_text} "
                f"ema_total={ema_total:.4f} "
                f"positions/s="
                f"{seen / elapsed:,.0f} "
                f"prep="
                f"{100 * prep_seconds / elapsed:.1f}%"
            )

fallback = [
    str(w.message)
    for w in caught
    if "fall back"
    in str(w.message).lower()
]

if fallback:
    raise RuntimeError(
        "DirectML fallback: "
        + " | ".join(fallback)
    )

pd.DataFrame(
    history
).to_csv(
    HISTORY_PATH,
    index=False,
)

torch.save({
    "step": STEPS,
    "model_state_dict": {
        k: v.detach().cpu()
        for k, v
        in model.state_dict().items()
    },
    "architecture": (
        "bpe200_global_coarse_to_fine"
    ),
    "coarse_radii": (
        tuple(COARSE_RADII)
    ),
    "bpe_merges": BPE_MERGES,
    "merge_hash": MERGE_HASH,
    "cache_format_version": (
        CACHE_FORMAT_VERSION
    ),
    "cache_engine": CACHE_ENGINE,
    "global_flank": GLOBAL_FLANK,
    "global_length": GLOBAL_LENGTH,
    "global_token_rf": GLOBAL_TOKEN_RF,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "final_ema_by_radius": {
        int(k): float(v)
        for k, v
        in ema.items()
    },
    "final_ema_total": (
        float(ema_total)
    ),
}, CHECKPOINT_PATH)

print(
    "Elapsed min:",
    round(
        (
            time.perf_counter()
            - started
        )
        / 60,
        2,
    ),
)

print(
    "Final EMA by radius:",
    ema,
)

print(
    "Final EMA total:",
    ema_total,
)

print(
    "Checkpoint:",
    CHECKPOINT_PATH,
)


step=    1 r256=0.3328 r128=0.5944 r64=0.6465 r32=0.6771 total=0.5627 e256=0.3328 e128=0.5944 e64=0.6465 e32=0.6771 ema_total=0.5627 positions/s=131,945 prep=15.9%
step=  100 r256=0.2768 r128=0.4863 r64=0.7081 r32=0.7214 total=0.5481 e256=0.3543 e128=0.5269 e64=0.6064 e32=0.6499 ema_total=0.5344 positions/s=133,945 prep=16.5%
step=  200 r256=0.4743 r128=0.5658 r64=0.6176 r32=0.6638 total=0.5804 e256=0.3441 e128=0.4983 e64=0.5943 e32=0.6439 ema_total=0.5202 positions/s=138,341 prep=16.7%
step=  300 r256=0.3326 r128=0.4445 r64=0.6231 r32=0.6461 total=0.5116 e256=0.3427 e128=0.4959 e64=0.5973 e32=0.6417 ema_total=0.5194 positions/s=141,018 prep=16.8%
step=  400 r256=0.3296 r128=0.5039 r64=0.5762 r32=0.6065 total=0.5040 e256=0.3316 e128=0.5071 e64=0.5990 e32=0.6408 ema_total=0.5196 positions/s=142,733 prep=16.9%
step=  500 r256=0.1683 r128=0.3405 r64=0.4333 r32=0.4552 total=0.3493 e256=0.3118 e128=0.4845 e64=0.5641 e32=0.6064 ema_total=0.4917 positions/s=144,871 prep=17.0%
step=  600 r256=

In [16]:
# ============================================================
# FULL HOLDOUT — GLOBAL COARSE-TO-FINE DIAGNOSTIC
# ============================================================
# No sampling and no torch.linspace here.
# This is exactly the complete validation tile index.

validation_tiles = (
    validation_generator
    .tile_index
    .copy()
)

print("Tiles:", len(validation_tiles))
print(
    "Position-strand:",
    int(
        validation_tiles[
            "allowed_position_strand"
        ].sum()
    ),
)
print(
    "Positives:",
    int(
        validation_tiles[
            "positive_positions"
        ].sum()
    ),
)
print(
    "Contigs:",
    validation_tiles[
        "contig"
    ].nunique(),
)


Tiles: 47743
Position-strand: 83807486
Positives: 56205
Contigs: 3


In [17]:
# ============================================================
# FULL HOLDOUT VALIDATION
# EXACT AP + COARSE-TARGET AP + PEAK ALIGNMENT
# ============================================================

import numpy as np
from scipy.special import expit
import ml_project.epic_cnn.evaluation as epic_eval


class BinaryAPHistogram:
    def __init__(
        self,
        levels=100_001,
    ):
        self.levels = levels

        self.pos_hist = np.zeros(
            levels,
            dtype=np.int64,
        )

        self.neg_hist = np.zeros(
            levels,
            dtype=np.int64,
        )

    def add(
        self,
        score,
        target,
    ):
        score = np.asarray(
            score,
            dtype=np.float64,
        )

        target = np.asarray(
            target,
            dtype=bool,
        )

        if score.size == 0:
            return

        q = np.rint(
            np.clip(
                score,
                0.0,
                1.0,
            )
            * (self.levels - 1)
        ).astype(
            np.int32
        )

        if target.any():
            self.pos_hist += np.bincount(
                q[target],
                minlength=self.levels,
            )

        neg = ~target

        if neg.any():
            self.neg_hist += np.bincount(
                q[neg],
                minlength=self.levels,
            )

    def metrics(self):
        pos_desc = (
            self.pos_hist[::-1]
        )

        neg_desc = (
            self.neg_hist[::-1]
        )

        total_pos = int(
            pos_desc.sum()
        )

        total_neg = int(
            neg_desc.sum()
        )

        total = (
            total_pos
            + total_neg
        )

        if (
            total_pos == 0
            or total == 0
        ):
            return {
                "positions": total,
                "positives": total_pos,
                "prevalence": np.nan,
                "average_precision": np.nan,
                "ap_lift": np.nan,
            }

        tp = np.cumsum(
            pos_desc,
            dtype=np.int64,
        )

        fp = np.cumsum(
            neg_desc,
            dtype=np.int64,
        )

        denom = (
            tp + fp
        )

        precision = np.divide(
            tp,
            denom,
            out=np.zeros_like(
                tp,
                dtype=np.float64,
            ),
            where=denom > 0,
        )

        recall_increment = (
            pos_desc.astype(
                np.float64
            )
            / total_pos
        )

        ap = float(
            np.sum(
                precision
                * recall_increment
            )
        )

        prevalence = (
            total_pos
            / total
        )

        return {
            "positions": total,
            "positives": total_pos,
            "prevalence": prevalence,
            "average_precision": ap,
            "ap_lift": (
                ap / prevalence
                if prevalence > 0
                else np.nan
            ),
        }


PEAK_SEARCH_RADIUS = 384

PEAK_HIT_RADII = (
    4,
    8,
    16,
    32,
    64,
    128,
    256,
)

exact_acc = {
    radius: BinaryAPHistogram()
    for radius in COARSE_RADII
}

coarse_acc = {
    radius: BinaryAPHistogram()
    for radius in COARSE_RADII
}

peak_offsets = {
    radius: []
    for radius in COARSE_RADII
}


def dilate_numpy_binary(
    exact,
    radius,
):
    # exact shape: [B,2,1024]
    x = torch.from_numpy(
        exact.astype(
            np.float32
        )
    ).reshape(
        -1,
        1,
        TARGET_LENGTH,
    )

    y = F.max_pool1d(
        x,
        kernel_size=(
            2 * radius
            + 1
        ),
        stride=1,
        padding=radius,
    )

    return (
        y.reshape(
            exact.shape
        )
        .numpy()
        > 0
    )


def add_peak_alignment(
    score,
    mask,
    exact_positive,
    head_radius,
):
    usable_score = np.where(
        mask,
        score,
        -np.inf,
    )

    B = score.shape[0]

    for b in range(B):
        for strand in range(2):
            positive_idx = np.flatnonzero(
                exact_positive[
                    b,
                    strand,
                ]
            )

            positive_idx = positive_idx[
                (
                    positive_idx
                    >= PEAK_SEARCH_RADIUS
                )
                & (
                    positive_idx
                    < TARGET_LENGTH
                    - PEAK_SEARCH_RADIUS
                )
            ]

            row = usable_score[
                b,
                strand,
            ]

            for pos in positive_idx:
                lo = (
                    pos
                    - PEAK_SEARCH_RADIUS
                )

                hi = (
                    pos
                    + PEAK_SEARCH_RADIUS
                    + 1
                )

                local = row[
                    lo:hi
                ]

                rel_peak = int(
                    np.argmax(local)
                )

                peak_pos = (
                    lo
                    + rel_peak
                )

                peak_offsets[
                    head_radius
                ].append(
                    abs(
                        peak_pos
                        - int(pos)
                    )
                )


def evaluate_coarse_to_fine(
    model,
    tiles,
    batch_size=8,
    progress_every=100,
):
    positions = 0
    positives = 0

    started = (
        time.perf_counter()
    )

    was_training = (
        model.training
    )

    model.eval()

    try:
        with warnings.catch_warnings(
            record=True
        ) as caught:

            warnings.simplefilter(
                "always"
            )

            for batch_no, start in enumerate(
                range(
                    0,
                    len(tiles),
                    batch_size,
                ),
                1,
            ):
                selected = tiles.iloc[
                    start:
                    start + batch_size
                ]

                batch = (
                    validation_generator
                    .make_batch(
                        selected
                    )
                )

                gb = (
                    build_cached_both_batch(
                        batch.tiles
                    )
                )

                outputs = forward_both(
                    model,
                    gb,
                    grad=False,
                )

                mask = (
                    batch.mask.numpy()
                    .astype(bool)
                )

                counts = (
                    batch.count.numpy()
                )

                exact_positive = (
                    (counts > 0)
                    & mask
                )

                for radius in COARSE_RADII:
                    pp, mp = outputs[
                        radius
                    ]

                    logits = np.stack(
                        (
                            pp.detach()
                            .cpu()
                            .numpy()[
                                :,
                                0,
                                :,
                            ],

                            mp.detach()
                            .cpu()
                            .numpy()[
                                :,
                                0,
                                ::-1,
                            ],
                        ),
                        axis=1,
                    )

                    score = expit(
                        logits
                    )

                    # Exact nucleotide AP:
                    exact_acc[
                        radius
                    ].add(
                        score[mask],
                        exact_positive[mask],
                    )

                    # Coarse target AP:
                    coarse_target = (
                        dilate_numpy_binary(
                            exact_positive,
                            radius,
                        )
                    )

                    # Ignore tile edges because coarse labels there
                    # would need positives from neighboring tiles.
                    coarse_valid = (
                        mask.copy()
                    )

                    coarse_valid[
                        ...,
                        :radius
                    ] = False

                    coarse_valid[
                        ...,
                        -radius:
                    ] = False

                    coarse_acc[
                        radius
                    ].add(
                        score[
                            coarse_valid
                        ],
                        coarse_target[
                            coarse_valid
                        ],
                    )

                    add_peak_alignment(
                        score,
                        mask,
                        exact_positive,
                        radius,
                    )

                positions += int(
                    mask.sum()
                )

                positives += int(
                    exact_positive.sum()
                )

                finished = (
                    start
                    + len(selected)
                    >= len(tiles)
                )

                if (
                    batch_no == 1
                    or batch_no
                    % progress_every
                    == 0
                    or finished
                ):
                    elapsed = (
                        time.perf_counter()
                        - started
                    )

                    print(
                        f"tiles="
                        f"{start + len(selected):>5}"
                        f"/{len(tiles)} "
                        f"positions="
                        f"{positions:>9} "
                        f"positions/s="
                        f"{positions / elapsed:,.0f}"
                    )

        fallback = tuple(
            str(w.message)
            for w in caught
            if "fall back"
            in str(w.message).lower()
        )

    finally:
        model.train(
            was_training
        )

    if fallback:
        raise RuntimeError(
            "DirectML fallback: "
            + " | ".join(
                fallback
            )
        )

    assert positions == int(
        tiles[
            "allowed_position_strand"
        ].sum()
    )

    assert positives == int(
        tiles[
            "positive_positions"
        ].sum()
    )

    exact_rows = []
    coarse_rows = []
    peak_summary_rows = []
    peak_hit_rows = []

    for radius in COARSE_RADII:
        exact_row = (
            exact_acc[
                radius
            ].metrics()
        )

        exact_row[
            "head_radius_bp"
        ] = radius

        exact_rows.append(
            exact_row
        )

        coarse_row = (
            coarse_acc[
                radius
            ].metrics()
        )

        coarse_row[
            "head_radius_bp"
        ] = radius

        coarse_rows.append(
            coarse_row
        )

        offsets = np.asarray(
            peak_offsets[
                radius
            ],
            dtype=np.int32,
        )

        if offsets.size:
            peak_summary_rows.append({
                "head_radius_bp": radius,
                "evaluated_true_tss": int(
                    offsets.size
                ),
                "search_radius_bp": (
                    PEAK_SEARCH_RADIUS
                ),
                "median_abs_offset_bp": float(
                    np.median(
                        offsets
                    )
                ),
                "p75_abs_offset_bp": float(
                    np.percentile(
                        offsets,
                        75,
                    )
                ),
                "p90_abs_offset_bp": float(
                    np.percentile(
                        offsets,
                        90,
                    )
                ),
            })

            for hit_radius in PEAK_HIT_RADII:
                peak_hit_rows.append({
                    "head_radius_bp": radius,
                    "hit_radius_bp": hit_radius,
                    "hit_fraction": float(
                        np.mean(
                            offsets
                            <= hit_radius
                        )
                    ),
                    "evaluated_true_tss": int(
                        offsets.size
                    ),
                })

    exact_df = pd.DataFrame(
        exact_rows
    )[[
        "head_radius_bp",
        "positions",
        "positives",
        "prevalence",
        "average_precision",
        "ap_lift",
    ]]

    coarse_df = pd.DataFrame(
        coarse_rows
    )[[
        "head_radius_bp",
        "positions",
        "positives",
        "prevalence",
        "average_precision",
        "ap_lift",
    ]]

    peak_summary_df = pd.DataFrame(
        peak_summary_rows
    )

    peak_hits_df = pd.DataFrame(
        peak_hit_rows
    )

    elapsed = (
        time.perf_counter()
        - started
    )

    return (
        exact_df,
        coarse_df,
        peak_summary_df,
        peak_hits_df,
        elapsed,
        positions / elapsed,
    )


checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

assert (
    checkpoint[
        "architecture"
    ]
    == "bpe200_global_coarse_to_fine"
)

assert tuple(
    checkpoint[
        "coarse_radii"
    ]
) == tuple(
    COARSE_RADII
)

assert (
    checkpoint[
        "bpe_merges"
    ]
    == BPE_MERGES
)

assert (
    checkpoint[
        "merge_hash"
    ]
    == MERGE_HASH
)

assert (
    checkpoint[
        "cache_format_version"
    ]
    == CACHE_FORMAT_VERSION
)

validation_model = (
    create_model()
    .to(device)
)

validation_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

(
    exact_metrics,
    coarse_metrics,
    peak_summary,
    peak_hits,
    seconds,
    speed,
) = evaluate_coarse_to_fine(
    validation_model,
    validation_tiles,
    batch_size=8,
    progress_every=500,
)

EXACT_PATH = (
    RUN_DIR
    / (
        f"exact_ap_by_head_"
        f"{STEPS}_steps.csv"
    )
)

COARSE_PATH = (
    RUN_DIR
    / (
        f"coarse_ap_by_head_"
        f"{STEPS}_steps.csv"
    )
)

PEAK_SUMMARY_PATH = (
    RUN_DIR
    / (
        f"peak_alignment_summary_"
        f"{STEPS}_steps.csv"
    )
)

PEAK_HITS_PATH = (
    RUN_DIR
    / (
        f"peak_alignment_hits_"
        f"{STEPS}_steps.csv"
    )
)

exact_metrics.to_csv(
    EXACT_PATH,
    index=False,
)

coarse_metrics.to_csv(
    COARSE_PATH,
    index=False,
)

peak_summary.to_csv(
    PEAK_SUMMARY_PATH,
    index=False,
)

peak_hits.to_csv(
    PEAK_HITS_PATH,
    index=False,
)

print()
print(
    "=== EXACT NUCLEOTIDE AP BY HEAD ==="
)
display(
    exact_metrics
)

print()
print(
    "=== COARSE-TARGET AP BY HEAD ==="
)
display(
    coarse_metrics
)

print()
print(
    "=== PEAK ALIGNMENT SUMMARY ==="
)
display(
    peak_summary
)

print()
print(
    "=== PEAK HIT FRACTIONS ==="
)
display(
    peak_hits
)

print()
print(
    "Validation min:",
    round(
        seconds / 60,
        2,
    ),
)

print(
    "Positions/s:",
    f"{speed:,.0f}",
)

print(
    "Exact AP:",
    EXACT_PATH,
)

print(
    "Coarse AP:",
    COARSE_PATH,
)

print(
    "Peak summary:",
    PEAK_SUMMARY_PATH,
)

print(
    "Peak hits:",
    PEAK_HITS_PATH,
)


tiles=    8/47743 positions=    11484 positions/s=372,390
tiles= 4000/47743 positions=  6861742 positions/s=579,297
tiles= 8000/47743 positions= 14166170 positions/s=601,012
tiles=12000/47743 positions= 21458144 positions/s=610,128
tiles=16000/47743 positions= 28627148 positions/s=612,080
tiles=20000/47743 positions= 35584246 positions/s=609,094
tiles=24000/47743 positions= 42571942 positions/s=606,287
tiles=28000/47743 positions= 49714988 positions/s=607,254
tiles=32000/47743 positions= 56582858 positions/s=605,402
tiles=36000/47743 positions= 63201944 positions/s=601,408
tiles=40000/47743 positions= 70164960 positions/s=600,618
tiles=44000/47743 positions= 77380732 positions/s=602,248
tiles=47743/47743 positions= 83807486 positions/s=601,377

=== EXACT NUCLEOTIDE AP BY HEAD ===


,head_radius_bp,positions,positives,prevalence,average_precision,ap_lift
0,256,83807486,56205,0.000671,0.005519,8.229409
1,128,83807486,56205,0.000671,0.005403,8.056051
2,64,83807486,56205,0.000671,0.005488,8.182888
3,32,83807486,56205,0.000671,0.005500,8.201048



=== COARSE-TARGET AP BY HEAD ===


,head_radius_bp,positions,positives,prevalence,average_precision,ap_lift
0,256,41884258,4173631,0.099647,0.147920,1.484439
1,128,62840668,3626613,0.057711,0.104849,1.816785
2,64,73326586,2426024,0.033085,0.074240,2.243899
3,32,78566126,1493655,0.019011,0.052925,2.783872



=== PEAK ALIGNMENT SUMMARY ===


,head_radius_bp,evaluated_true_tss,search_radius_bp,median_abs_offset_bp,p75_abs_offset_bp,p90_abs_offset_bp
0,256,13974,384,118.0,236.0,329.0
1,128,13974,384,116.0,235.0,330.0
2,64,13974,384,116.0,235.0,329.0
3,32,13974,384,117.0,235.0,331.0



=== PEAK HIT FRACTIONS ===


,head_radius_bp,hit_radius_bp,hit_fraction,evaluated_true_tss
0,256,4,0.057178,13974
1,256,8,0.102333,13974
2,256,16,0.168169,13974
3,256,32,0.249105,13974
4,256,64,0.357163,13974
5,256,128,0.523043,13974
6,256,256,0.783527,13974
7,128,4,0.059110,13974
8,128,8,0.106555,13974
9,128,16,0.176685,13974



Validation min: 2.32
Positions/s: 601,310
Exact AP: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP017_BPE200_GLOBAL_COARSE_TO_FINE\run_001\exact_ap_by_head_10000_steps.csv
Coarse AP: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP017_BPE200_GLOBAL_COARSE_TO_FINE\run_001\coarse_ap_by_head_10000_steps.csv
Peak summary: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP017_BPE200_GLOBAL_COARSE_TO_FINE\run_001\peak_alignment_summary_10000_steps.csv
Peak hits: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP017_BPE200_GLOBAL_COARSE_TO_FINE\run_001\peak_alignment_hits_10000_steps.csv
